In [ ]:
from collections import defaultdict


# first attempt
class Tokenizer:
    def __init__(self) -> None:
        self.merges: defaultdict[tuple[int, int], int] = defaultdict(int)
        self.count: defaultdict[tuple[int, int], int] = defaultdict(int)
        self.vocab = {x: bytes([x]) for x in range(256)}

    def encode(self, text: str):
        byted_text = text.encode("utf-8")
        indices = list(byted_text)
        merges = self.merge(indices, 3)
        return merges

    def merge(self, indices: list[int], merge_num):
        for i in range(merge_num):
            for idx, idx2 in zip(indices, indices[1:]):
                self.count[(idx, idx2)] = self.count[(idx, idx2)] + 1
            max_pair = max(self.count, key=lambda pos: self.count[pos])
            new_idx = 255 + i
            self.merges[max_pair] = new_idx
            self.vocab[new_idx] = self.vocab[max_pair[0]] + self.vocab[max_pair[1]]
            new_indices = []
            j = 0
            while j < len(indices):
                if (
                    j < len(indices) - 1
                    and indices[j] == max_pair[0]
                    and indices[j + 1] == max_pair[1]
                ):
                    new_indices.append(new_idx)
                    j += 2
                else:
                    new_indices.append(indices[j])
                    j += 1
            indices = new_indices
        return indices

    def decode(self, indices: list[int]):
        s = ""
        for idx in indices:
            byted_char = self.vocab[idx]
            s += bytes(byted_char).decode("utf-8")
        return s


tokenizer = Tokenizer()
encoded_text = tokenizer.encode("This is some text")
tokenizer.decode(encoded_text)


'This is some text'

In [44]:
from dataclasses import dataclass


# second attempt
@dataclass
class TokenizerParams:
    vocab: dict[int, bytes]
    merges: dict[tuple[int, int], int]


class Tokenizer:
    def __init__(self, params: TokenizerParams):
        self.params = params

    def encode(self, text: str) -> list[int]:
        i = 0
        indices = list(text.encode("utf-8"))
        new_indices = []
        n = len(indices)
        while i < n:
            if i < n - 1 and (indices[i], indices[i + 1]) in self.params.merges:
                new_indices.append(self.params.merges[(indices[i], indices[i + 1])])
                i += 2
            else:
                new_indices.append(indices[i])
                i += 1
        return new_indices

    def decode(self, indices: list[int]) -> str:
        byte = list(map(self.params.vocab.get, indices))
        return b"".join(byte).decode("utf-8")

    def train(self, corpus: str, num_merge: int):
        byted_corpus = corpus.encode("utf-8")
        indices = list(byted_corpus)
        for i in range(num_merge):
            count = defaultdict(int)
            for idx1, idx2 in zip(indices, indices[1:]):
                count[(idx1, idx2)] += 1
            pair = max(count, key=count.get)
            new_index = 255 + i
            self.params.merges[pair] = new_index
            self.params.vocab[new_index] = (
                self.params.vocab[pair[0]] + self.params.vocab[pair[1]]
            )

            indices = self.merge(indices, pair, new_index)
        return byted_corpus

    def merge(
        self, indices: list[int], pair: tuple[int, int], new_index: int
    ) -> list[int]:
        new_indices = []
        n = len(indices)
        i = 0
        while i < n:
            if i < n - 1 and indices[i] == pair[0] and indices[i + 1] == pair[1]:
                new_indices.append(new_index)
                i += 2
            else:
                new_indices.append(indices[i])
                i += 1
        return new_indices


init_vocab = {x: bytes([x]) for x in range(256)}
init_merges = {}
train_corpus = "This is some text"
test_text = train_corpus
params = TokenizerParams(vocab=init_vocab, merges=init_merges)
tokenizer = Tokenizer(params)
tokenizer.train(corpus=train_corpus, num_merge=3)
encoded_text = tokenizer.encode(test_text)
decoded_text = tokenizer.decode(encoded_text)
print(decoded_text)


This is some text


In [ ]:
list(zip([1, 2], [3, 4, 5]))
list(
    zip(
        [1, 2],
        [
            3,
        ],
    )
)

[(1, 3)]

In [55]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")
text = "你好世界"
token_ids = tokenizer.encode(text)

for tid in token_ids:
    print(f"{tid} → '{tokenizer.decode([tid])}'")

You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


109266 → '你好'
96748 → '世界'
